# Modelos de predicción clusters

In [1]:
import pandas as pd
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import recall_score, confusion_matrix
import joblib

def convert_binary_columns(df: pd.DataFrame):
    """
    Detecta en el DataFrame todas las columnas con exactamente dos valores únicos,
    las convierte a booleano (pandas BooleanDtype) y devuelve, además, un diccionario
    con el mapeo de qué valor original se asocia a True y a False.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame de entrada.

    Retorna
    -------
    df_bool : pd.DataFrame
        Copia del DataFrame con las columnas binarias convertidas a booleanas.
    mapping : dict
        Diccionario cuya clave es el nombre de cada columna convertida y cuyo valor
        es otro dict {'true': valor_verdadero, 'false': valor_falso}.
    """
    df_bool = df.copy()
    mapping = {}

    for col in df_bool.columns:
        # obtenemos los valores únicos (ignorando NaN)
        uniques = df_bool[col].dropna().unique()
        if len(uniques) == 2:
            # ordenamos para asignar el mayor/segundo a True por convención
            val_false, val_true = sorted(uniques)
            # aplicamos el mapeo y convertimos a BooleanDtype de pandas
            df_bool[col] = df_bool[col].map({val_false: False, val_true: True}).astype("boolean")
            # guardamos en el diccionario qué valor original es True/False
            mapping[col] = {"true": val_true, "false": val_false}

    return df_bool, mapping



def DT_vict_perp_classifier(
    df: pd.DataFrame,
    target_col: str = 'VICTIMA_PERPETRADOR',
    output_dir: str = 'vict_perp_classifier',
    test_size: float = 0.2,
    random_state: int = 42,
    max_depth: int = 8
):
    """
    Entrena un DecisionTreeClassifier con profundidad máxima de `max_depth` para predecir la clase 1 en `target_col`,
    maximizando el recall de la categoría 1 y evitando overfitting mediante cost-complexity pruning.
    Genera artefactos: CSVs, gráficas, árbol podado, reglas, modelo, tabla de métricas y matriz de confusión.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con variables predictoras y columna objetivo.
    target_col : str
        Nombre de la columna objetivo binaria (0/1).
    output_dir : str
        Carpeta donde guardar los artefactos.
    test_size : float
        Proporción del set de test.
    random_state : int
        Semilla para reproducibilidad.
    max_depth : int
        Profundidad máxima permitida para el árbol.

    Retorna
    -------
    pruned_clf : DecisionTreeClassifier
        Árbol podado con mejor ccp_alpha y profundidad ≤ max_depth.
    best_alpha : float
        Valor de ccp_alpha seleccionado.
    recall_1_final : float
        Recall de la clase 1 en test.
    recall_0_final : float
        Recall de la clase 0 en test.
    """
    # 0. Crear carpeta de salida
    os.makedirs(output_dir, exist_ok=True)

    # 1. Definir X e y
    y = df[target_col].astype(int)
    X = df.drop(columns=[target_col])
    feature_names = X.columns.tolist()

    # 2. División estratificada
    idx_train, idx_test = train_test_split(
        df.index,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )
    X_train = X.loc[idx_train].values
    X_test  = X.loc[idx_test].values
    y_train = y.loc[idx_train].values
    y_test  = y.loc[idx_test].values

    # Guardar splits
    X.loc[idx_train].to_csv(os.path.join(output_dir, 'X_train.csv'), index=True)
    X.loc[idx_test].to_csv(os.path.join(output_dir, 'X_test.csv'), index=True)
    y.loc[idx_train].to_csv(os.path.join(output_dir, 'y_train.csv'), index=True)
    y.loc[idx_test].to_csv(os.path.join(output_dir, 'y_test.csv'), index=True)

    # 3. Entrenar árbol completo con profundidad máxima
    full_clf = DecisionTreeClassifier(max_depth=max_depth, random_state=random_state)
    full_clf.fit(X_train, y_train)

    # 4. Obtener cost-complexity pruning path
    path = full_clf.cost_complexity_pruning_path(X_train, y_train)
    ccp_alphas = path.ccp_alphas

    # 5. Evaluar recall de clase 1 en test para cada alpha
    test_recall_1 = []
    for alpha in ccp_alphas:
        clf = DecisionTreeClassifier(max_depth=max_depth, random_state=random_state, ccp_alpha=alpha)
        clf.fit(X_train, y_train)
        test_recall_1.append(recall_score(y_test, clf.predict(X_test), pos_label=1))

    plt.figure()
    plt.plot(ccp_alphas, test_recall_1, marker='o', drawstyle='steps-post')
    plt.xscale('log')
    plt.xlabel('ccp_alpha')
    plt.ylabel('Test Recall categoría 1')
    plt.title(f'Recall(1) vs ccp_alpha (max_depth={max_depth})')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'recall_vs_ccp_alpha.png'), dpi=300)
    plt.close()

    # 6. Selección del mejor alpha y poda final
    best_idx = int(np.argmax(test_recall_1))
    best_alpha = ccp_alphas[best_idx]
    pruned_clf = DecisionTreeClassifier(
        max_depth=max_depth,
        random_state=random_state,
        ccp_alpha=best_alpha
    )
    pruned_clf.fit(X_train, y_train)

    # 7. Predicciones y probabilidades
    y_pred = pruned_clf.predict(X_test)
    y_proba = pruned_clf.predict_proba(X_test)
    proba_pred = [probs[pred] for probs, pred in zip(y_proba, y_pred.astype(int))]

    df_res = pd.DataFrame({
        'ori_idx': idx_test,
        'real_value': y_test,
        'predicted_value': y_pred,
        'probability': proba_pred
    })
    df_res.to_csv(os.path.join(output_dir, 'predictions_with_proba.csv'), index=False)

    # 8. Métricas finales y tabla
    recall_1_final = recall_score(y_test, y_pred, pos_label=1)
    recall_0_final = recall_score(y_test, y_pred, pos_label=0)
    metrics_df = pd.DataFrame({
        'class': [0, 1],
        'recall': [recall_0_final, recall_1_final]
    })
    metrics_csv = os.path.join(output_dir, 'metrics.csv')
    metrics_df.to_csv(metrics_csv, index=False)

    # 9. Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=['true_0', 'true_1'],
        columns=['pred_0', 'pred_1']
    )
    cm_csv = os.path.join(output_dir, 'confusion_matrix.csv')
    cm_df.to_csv(cm_csv, index=True)

    print(f"Profundidad máxima: {max_depth}")
    print(f"Mejor ccp_alpha: {best_alpha:.5f}")
    print("Tabla de métricas finales:")
    print(metrics_df.to_string(index=False))
    print("Matriz de confusión:")
    print(cm_df.to_string())
    print(f"Métricas guardadas en: {metrics_csv}")
    print(f"Matriz de confusión guardada en: {cm_csv}")

    # 10. Visualizar árbol podado
    plt.figure(figsize=(20, 10))
    plot_tree(
        pruned_clf,
        filled=True,
        feature_names=feature_names,
        class_names=['0', '1'],
        rounded=True
    )
    plt.title(f"Árbol podado (max_depth={max_depth}, ccp_alpha={best_alpha:.5f})")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'pruned_tree.png'), dpi=300)
    plt.close()

    # 11. Exportar reglas
    rules = export_text(pruned_clf, feature_names=feature_names)
    with open(os.path.join(output_dir, 'tree_rules.txt'), 'w', encoding='utf-8') as f:
        f.write(rules)

    # 12. Guardar modelo
    joblib.dump(pruned_clf, os.path.join(output_dir, 'pruned_clf.joblib'))

    return pruned_clf, best_alpha, recall_1_final, recall_0_final


In [2]:
df_clust_vict = pd.read_csv('/content/clust_vict.csv')
df_clust_perp = pd.read_csv('/content/clust_per.csv')
df_global_vars = pd.read_csv('/content/global_vars.csv')
df_target_vars = pd.read_csv('/content/target_vars.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/clust_vict.csv'

In [ ]:
df_clust_vict.rename(columns={'cluster': 'clust_vict'}, inplace=True)
df_clust_perp.rename(columns={'cluster': 'clust_perp'}, inplace=True)

## Predicción de grupo

 - Buscamos a partir de las variables globales.
 - Predecir el cluster de víctima al que puede pertenecer.
 - Predecir el cluster de perpetrador al que puede pertenecer.

In [ ]:
df_global_vict_clust = df_global_vars.join([df_target_vars['VÍCTIMA'],df_clust_vict["clust_vict"]],how="inner")
df_global_perp_clust = df_global_vars.join(df_target_vars['PERPETRADOR'],df_clust_perp["clust_perp"],how="inner")

In [ ]:
df_vic_clust = df_global_vict_clust[~((df_global_vict_clust["GENERO_BIN_2"] == 1) | (df_global_vict_clust["ORIENTSEX.BN_3"] == 1))].drop(columns=["GENERO_BIN_2","ORIENTSEX.BN_3","CONVIVEN.6"]).reset_index(drop=True)
df_per_clust = df_global_perp_clust[~((df_global_perp_clust["GENERO_BIN_2"] == 1) | (df_global_perp_clust["ORIENTSEX.BN_3"] == 1))].drop(columns=["GENERO_BIN_2","ORIENTSEX.BN_3","CONVIVEN.6"]).reset_index(drop=True)

### Predicción Clúster Víctima

 - Hemos eliminado los registros de las categorías 0 y 3 que eran las mayoritarias y las más desbalanceadas.
 - Tendremos dos modelos una que nos dira a que cluster es más probable que pertenezca del resto de clusteres.
 - Otra red que decidira entre el cluster 0 y 3 cual es el más probable que pertenezca.
 - Así podremos decir que cuando predecimos una potencial víctima le podremos asignar a esta dos posibles perfiles probables en base a las variables.

In [ ]:
clf_full, recall_macro_full, recall_per_class_full = DT_multiclass_full_overfit(
    df=df_vic_clust,
    target_col='clust_vict',
    output_dir='DT_multiclassifier_full_overfit_1245678',
    random_state=42
)

print(f"Recall macro sobre entrenamiento: {recall_macro_full:.4f}")
print("Recall por clase:", recall_per_class_full)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import recall_score, confusion_matrix
import joblib

def DT_multiclass_full_overfit(
    df: pd.DataFrame,
    target_col: str = 'TARGET',
    output_dir: str = 'multiclass_full_overfit',
    random_state: int = 42
):
    """
    Entrena un DecisionTreeClassifier sin límite de profundidad (max_depth=None)
    usando **todo** el DataFrame para entrenamiento, de modo que sobreajuste perfectamente.
    Genera artefactos: CSVs de predicción y métricas, visualización del árbol,
    reglas y modelo serializado.

    Retorna
    -------
    clf : DecisionTreeClassifier
        Árbol completamente entrenado (overfit).
    recall_macro : float
        Recall macro calculado sobre el mismo conjunto de entrenamiento (debería ser 1.0).
    recall_per_class : dict
        Recall de cada clase (también 1.0 por clase).
    """
    # Crear carpeta de salida
    os.makedirs(output_dir, exist_ok=True)
    df = df[~df['clust_vict'].isin([0, 3])].reset_index(drop=True)
    # Separar X, y
    y = df[target_col].astype(int)
    X = df.drop(columns=[target_col])
    feature_names = X.columns.tolist()

    # Entrenar sobre todo el dataset sin límites
    clf = DecisionTreeClassifier(
        criterion='gini',
        splitter='best',
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        ccp_alpha=0.0,
        random_state=random_state
    )
    clf.fit(X.values, y.values)

    # Predicción sobre el mismo conjunto
    y_pred = clf.predict(X.values)

    # Guardar predicciones
    pd.DataFrame({
        'real_value': y.values,
        'predicted_value': y_pred
    }).to_csv(
        os.path.join(output_dir, 'predictions_full.csv'),
        index=False
    )

    # Métricas
    classes = clf.classes_
    recall_vals = recall_score(y.values, y_pred, labels=classes, average=None)
    recall_macro = recall_score(y.values, y_pred, average='macro')
    metrics_df = pd.DataFrame({
        'class': classes,
        'recall': recall_vals
    })
    metrics_df.to_csv(
        os.path.join(output_dir, 'metrics_full.csv'),
        index=False
    )

    # Matriz de confusión
    cm = confusion_matrix(y.values, y_pred, labels=classes)
    pd.DataFrame(
        cm,
        index=[f'true_{c}' for c in classes],
        columns=[f'pred_{c}' for c in classes]
    ).to_csv(
        os.path.join(output_dir, 'confusion_matrix_full.csv')
    )

    # Visualizar el árbol completo
    plt.figure(figsize=(20, 10))
    plot_tree(
        clf,
        filled=True,
        feature_names=feature_names,
        class_names=[str(c) for c in classes],
        rounded=True
    )
    plt.title("Árbol completo (overfit)")
    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, 'tree_full.png'),
        dpi=300
    )
    plt.close()

    # Reglas y modelo
    with open(
        os.path.join(output_dir, 'rules_full.txt'),
        'w', encoding='utf-8'
    ) as f:
        f.write(export_text(clf, feature_names=feature_names))
    joblib.dump(clf, os.path.join(output_dir, 'model_full.joblib'))

    return clf, recall_macro, dict(zip(classes.tolist(), recall_vals.tolist()))


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix, classification_report
import joblib
from tensorflow.keras import models, layers, callbacks
from tensorflow.keras.metrics import Recall


def NN_vict_clust_03(
    df: pd.DataFrame,
    output_dir: str = 'nn_vict_clust',
    test_size: float = 0.1,
    val_size: float = 0.1,
    random_state: int = 42,
    epochs: int = 1000,
    batch_size: int = 32,
    dropout_rate: float = 0.3
):
    """
    Entrena una red neuronal multicategoría optimizada para maximizar el recall por clase.
    """
    os.makedirs(output_dir, exist_ok=True)

    # 1. Filtrar clases 0 y 3
    df = df[df['clust_vict'].isin([0, 3])].reset_index(drop=True)

    # 2. Preparar etiquetas (reencode de 0..K-1)
    y_raw = df['clust_vict'].values
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    num_classes = len(le.classes_)

    # 3. Preparar características
    X = df.drop(columns=['clust_vict']).values

    # 4. División estratificada en train/val/test
    idx = np.arange(len(y))
    idx_train_val, idx_test = train_test_split(
        idx, test_size=test_size,
        random_state=random_state, stratify=y
    )
    idx_train, idx_val = train_test_split(
        idx_train_val, test_size=val_size,
        random_state=random_state, stratify=y[idx_train_val]
    )

    # 5. One-hot encoding de etiquetas
    y_train = np.eye(num_classes)[y[idx_train]]
    y_val   = np.eye(num_classes)[y[idx_val]]
    y_test  = np.eye(num_classes)[y[idx_test]]

    # 6. Escalado de características
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[idx_train])
    X_val   = scaler.transform(X[idx_val])
    X_test  = scaler.transform(X[idx_test])
    joblib.dump(scaler, os.path.join(output_dir, 'scaler.joblib'))

    # 7. Pesos de clase
    weights = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=np.arange(num_classes),
        y=y[idx_train]
    )
    class_weights = {i: w for i, w in enumerate(weights)}

    # 8. Definir modelo
    model = models.Sequential([
        layers.Input(shape=(X.shape[1],)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(128, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(64, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(16, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(num_classes, activation='softmax')
    ])
    recall_metrics = [Recall(class_id=i, name=f'recall_{i}') for i in range(num_classes)]
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy', *recall_metrics]
    )

    # 9. Callback para guardar el mejor modelo según recall de la clase 0 (o la que elijas)
    monitor_metric = 'val_recall_0'
    checkpoint = callbacks.ModelCheckpoint(
        filepath=os.path.join(output_dir, 'best_model.h5'),
        monitor=monitor_metric,
        mode='max',
        save_best_only=True,
        verbose=1
    )

    # 10. Entrenamiento
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weights,
        callbacks=[checkpoint],
        verbose=2
    )

    # 11. Evaluación en test
    test_loss, test_acc, *test_recalls = model.evaluate(X_test, y_test, verbose=0)
    metrics = {'test_loss': test_loss, 'test_accuracy': test_acc}
    for i, r in enumerate(test_recalls):
        metrics[f'recall_{i}'] = r
    with open(os.path.join(output_dir, 'metrics.txt'), 'w') as f:
        for k, v in metrics.items():
            f.write(f"{k}: {v:.4f}\n")

    # 12. Matriz de confusión y reporte
    y_pred = np.argmax(model.predict(X_test), axis=1)
    y_true = y[idx_test]
    cm = confusion_matrix(y_true, y_pred)
    pd.DataFrame(cm,
                 index=[f'true_{c}' for c in le.classes_],
                 columns=[f'pred_{c}' for c in le.classes_])\
      .to_csv(os.path.join(output_dir, 'confusion_matrix.csv'))

    report = classification_report(y_true, y_pred, output_dict=True, target_names=[str(c) for c in le.classes_])
    pd.DataFrame(report).transpose()\
      .to_csv(os.path.join(output_dir, 'classification_report.csv'))

    return model, history, metrics, le

In [ ]:
# Entrenar la red y guardar artefactos en 'output_nn'
model, history, metrics, le = NN_vict_clust_03(
    df_vic_clust,
    output_dir='output_nn_03',
    test_size=0.2,
    val_size=0.2,
    random_state=123,
    epochs=10,
    batch_size=32,
    dropout_rate=0.3
)
# Mostrar métricas de test
print("Métricas de test:", metrics)

### Predicción Clúster Perpetrador

 - Hemos eliminado los registros de las categorías 0 y 3 que eran las mayoritarias y las más desbalanceadas.
 - Tendremos dos modelos una que nos dira a que cluster es más probable que pertenezca del resto de clusteres.
 - Otra red que decidira entre el cluster 0 y 3 cual es el más probable que pertenezca.
 - Así podremos decir que cuando predecimos una potencial víctima le podremos asignar a esta dos posibles perfiles probables en base a las variables.

## Predicción VICTIMA_PERPETRADOR

 - Si disponemos que perfil de perpretador y de víctima tiene el usuario.
 - Junto con las variables globales.
 - Somos capaces de construir un sistema de reglas que nos predice el potencial de ser victima y agresor del usuario con un un muy alto porcentaje de acierto.

In [ ]:
df_features = df_global_vars.join([
    df_clust_vict["clust_vict"],
    df_clust_perp["clust_perp"],
],how="inner")
df_features.head()

In [ ]:
df = df_features.join(df_target_vars['VICTIMA_PERPETRADOR'],how="inner")
df.head()

In [ ]:
df = df[~((df["GENERO_BIN_2"] == 1) | (df["ORIENTSEX.BN_3"] == 1))].drop(columns=["GENERO_BIN_2","ORIENTSEX.BN_3","CONVIVEN.6"]).reset_index(drop=True)

In [ ]:
df_encoded = pd.get_dummies(
    df,
    columns=['clust_vict', 'clust_perp'],
    prefix=['vict', 'perp'],
    prefix_sep='_'
)
df_encoded.head()

In [ ]:
f_df,map = convert_binary_columns(df_encoded)

In [ ]:
pruned_model, best_alpha, recall_1_final, recall_0_final = DT_vict_perp_classifier(
    df=f_df,
    target_col='VICTIMA_PERPETRADOR',
    output_dir='vict_perp_classifier_output',
    test_size=0.2,
    random_state=42
)